# C1 ML Training Notebook

This notebook loads the C1 dataset, cleans it, trains an XGBoost model, evaluates it, and saves the model artifacts.

In [4]:
import json
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from xgboost import XGBClassifier

In [16]:
def _resolve_base_dir():
    workspace_hint = r"D:\\R26-CS-003\\R26-CS-003"
    if os.path.exists(os.path.join(workspace_hint, 'core', 'c1', 'data', 'master_dataset.csv')):
        return workspace_hint
    candidates = [os.getcwd()] + [os.path.abspath(os.path.join(os.getcwd(), *(['..'] * i))) for i in range(1, 6)]
    for base in candidates:
        probe = os.path.join(base, 'core', 'c1', 'data', 'master_dataset.csv')
        if os.path.exists(probe):
            return base
    return os.getcwd()

BASE_DIR = _resolve_base_dir()
DATA_PATH = os.path.join(BASE_DIR, 'core', 'c1', 'data', 'master_dataset.csv')
CLEAN_PATH = os.path.join(BASE_DIR, 'core', 'c1', 'data', 'dataset_clean.csv')
FEATURES_PATH = os.path.join(BASE_DIR, 'core', 'c1', 'data', 'dataset_clean_features.json')
MODEL_PATH = os.path.join(BASE_DIR, 'core', 'c1', 'models', 'extension_detector_model.pkl')
META_PATH = os.path.join(BASE_DIR, 'core', 'c1', 'models', 'training_meta.json')
IMPORTANCE_PATH = os.path.join(BASE_DIR, 'core', 'c1', 'models', 'feature_importance.csv')
LABEL_COL = 'label'

print('BASE_DIR:', BASE_DIR)
print('DATA_PATH:', DATA_PATH)

BASE_DIR: D:\\R26-CS-003\\R26-CS-003
DATA_PATH: D:\\R26-CS-003\\R26-CS-003\core\c1\data\master_dataset.csv


In [23]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('Columns (first 10):', df.columns[:10].tolist())
print('Label counts:', df[LABEL_COL].value_counts().to_dict())

Shape: (946, 68)
Columns (first 10): ['label', 'md5', 'sha1', 'sha256', 'size', 'type', 'mimetype', 'name', 'crc32', 'sha512']
Label counts: {'benign': 924, 'malware': 22}


In [19]:
def normalize_label(value):
    if value is None:
        return None
    raw = str(value).strip().lower()
    if raw in {'1', 'malicious', 'malware', 'bad'}:
        return 1
    if raw in {'0', 'benign', 'safe', 'good'}:
        return 0
    return None

df[LABEL_COL] = df[LABEL_COL].apply(normalize_label)
df = df.dropna(subset=[LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# Convert feature columns to numeric
for col in [c for c in df.columns if c != LABEL_COL]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop non-numeric columns if any remain
non_numeric = df.drop(columns=[LABEL_COL]).select_dtypes(exclude=['number']).columns
if len(non_numeric) > 0:
    df = df.drop(columns=list(non_numeric))

df = df.replace([float('inf'), float('-inf')], pd.NA)
df = df.fillna(0)

# Clamp to float32 range and cast
feature_cols = [c for c in df.columns if c != LABEL_COL]
max_val = float(np.finfo('float32').max / 2)
df[feature_cols] = df[feature_cols].clip(lower=-max_val, upper=max_val)
df[feature_cols] = df[feature_cols].astype('float32')

# Reorder columns
df = df[feature_cols + [LABEL_COL]]

os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
with open(FEATURES_PATH, 'w', encoding='utf-8') as f:
    json.dump(feature_cols, f, indent=2)

print('Clean dataset saved:', CLEAN_PATH)
print('Label counts:', df[LABEL_COL].value_counts().to_dict())

Clean dataset saved: D:\\R26-CS-003\\R26-CS-003\core\c1\data\dataset_clean.csv
Label counts: {0: 924, 1: 22}


In [21]:
X = df.drop(columns=[LABEL_COL])
y = df[LABEL_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

benign_count = (y_train == 0).sum()
malicious_count = (y_train == 1).sum()
scale_weight = benign_count / max(malicious_count, 1)

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
 )

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print('Precision:', round(precision, 3))
print('Recall:', round(recall, 3))
print('F1:', round(f1, 3))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Precision: 1.0
Recall: 1.0
F1: 1.0

Classification report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00       186
   Malicious       1.00      1.00      1.00         4

    accuracy                           1.00       190
   macro avg       1.00      1.00      1.00       190
weighted avg       1.00      1.00      1.00       190

Confusion matrix:
[[186   0]
 [  0   4]]


In [22]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(model, MODEL_PATH)

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.to_csv(IMPORTANCE_PATH, header=['importance'])

meta = {
    'input': DATA_PATH,
    'label_col': LABEL_COL,
    'scale_pos_weight': float(scale_weight),
    'train_rows': int(X_train.shape[0]),
    'test_rows': int(X_test.shape[0]),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
}
with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('Model saved:', MODEL_PATH)
print('Feature importances saved:', IMPORTANCE_PATH)
print('Training meta saved:', META_PATH)

Model saved: D:\\R26-CS-003\\R26-CS-003\core\c1\models\extension_detector_model.pkl
Feature importances saved: D:\\R26-CS-003\\R26-CS-003\core\c1\models\feature_importance.csv
Training meta saved: D:\\R26-CS-003\\R26-CS-003\core\c1\models\training_meta.json
